# Street View Insights (`SV_PANO`): Identify, Locate & Describe Houses Along a Street

[Open in Colab](https://colab.research.google.com/github/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/blind_corridor_house_triangulation_and_verification.ipynb) · [View on GitHub](https://github.com/googlemaps-samples/insights-samples/blob/svi-gemini-improvements/street_view_insights/panoramic/notebooks/blind_corridor_house_triangulation_and_verification.ipynb)

This notebook scans a short street using **Street View Insights panoramic frames (`SV_PANO`)**, with no list of addresses as input, and answers three questions:

| Objective | Question | How it's measured here |
| :--- | :--- | :--- |
| 1. Multi-view identification | Can we find the same house in **3 or more** images? | Houses grouped with 3+ views vs. set aside |
| 2. Triangulation | Can we compute the house's **lat/lng** from those views? | Located houses + how tightly the rays meet |
| 3. Feature detection | Can we describe the house (walls, roof, garage, …)? | Features combined across views + agreement |

A few meters of location error, or a point near the edge of a building, is fine for these goals. The same pattern works for other assets (poles, signs, trees) by changing the Gemini schema.

| Step | What happens | Tools |
| :--- | :--- | :--- |
| 1. Select images | Pick one car drive along an ~80 m street | BigQuery |
| 2. Detect | Find each house's wall, front door, corners, house number, and features | Gemini on Vertex AI |
| 3. Group & triangulate | Turn pixels into compass rays, group views of the same house, intersect rays | NumPy / SciPy |
| 4. Sanity check | Is each point near a real building and a real address? | Overture Maps, Geocoding API |

Each step is explained in plain language right above its code. Everything runs live in your own project; nothing is pre-computed or bundled.

## 1. Before you start

**Enable these APIs** in the project you run from: BigQuery, Vertex AI, Cloud Storage, and **Geocoding API** (only for the optional address check in Step 4).

**IAM roles** for the user running the notebook:

| Role | Why |
| :--- | :--- |
| `roles/bigquery.jobUser` | Run queries in your project |
| `roles/bigquery.dataViewer` | Read your subscribed Street View Insights dataset |
| `roles/storage.objectViewer` | Read the Street View Insights image bucket |
| `roles/aiplatform.user` | Call Gemini on Vertex AI |
| `roles/serviceusage.serviceUsageConsumer` | Bill Geocoding API calls to your project |

**Approximate cost per run** (one 80 m street, ~60 frames): one BigQuery query (~2 GB scanned with `gcs_uri`), ~60 Gemini calls, and one Geocoding request per found house. The notebook prints a dry-run estimate before querying and a token count after Gemini.

**Terms & attribution:** Street View Insights imagery is governed by your Google Maps Platform agreement. Show **"Imagery © Google"** on figures, keep imagery in memory (this notebook never writes images to disk), and do not keep `gcs_uri` values beyond your snapshot retention window. Addresses from the Geocoding API must be shown with **Google Maps** attribution and used per the [Google Maps Platform Terms](https://cloud.google.com/maps-platform/terms). Overture building data is © Overture Maps Foundation and includes © OpenStreetMap contributors data under [ODbL](https://opendatacommons.org/licenses/odbl/).

In [ ]:
# Install the one package that is not pre-installed on Colab (the others are upgraded to recent versions).
%pip install -q --upgrade google-genai google-cloud-bigquery db-dtypes overturemaps shapely

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Imports & authentication
import concurrent.futures as cf
import dataclasses
from dataclasses import dataclass
import io
import json
import math
import os
import random
import sys
import time
import urllib.parse
from typing import Any, Mapping, Sequence

import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from google import genai
from google.genai import types
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field
from scipy.optimize import linear_sum_assignment
import shapely
from shapely.geometry import MultiPolygon, Point, Polygon

if "google.colab" in sys.modules:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()

print("Libraries imported.")

## 2. Configure your project and street

* `PROJECT_ID`: the project that runs queries and pays for Gemini / Geocoding.
* `SVI_PROJECT_ID` / `SVI_DATASET`: where your linked Street View Insights dataset lives (often the same project).
* `CORRIDOR_LAT`, `CORRIDOR_LNG`: the middle of the street you want to scan. Note that we only give a **street location**, never a house address.

In [ ]:
# @title Configuration
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
SVI_PROJECT_ID = ""  # @param {type:"string"}
SVI_DATASET = "imagery_insights___us"  # @param {type:"string"}
VERTEX_LOCATION = "global"  # @param {type:"string"}
GEMINI_MODEL = "gemini-4-argon"  # @param {type:"string"}

# Middle of the street segment to scan (example: a residential street in Lakeland, FL)
CORRIDOR_LAT = 28.05070  # @param {type:"number"}
CORRIDOR_LNG = -81.96126  # @param {type:"number"}
CORRIDOR_RADIUS_M = 45.0  # @param {type:"number"}
MAX_PANOS = 14  # @param {type:"integer"}

MAX_QUERY_GB = 5.0  # @param {type:"number"}
ENABLE_ADDRESS_CHECK = True  # @param {type:"boolean"}

# Rough Gemini price for the cost printout (USD per 1M tokens). Check current Vertex AI pricing.
GEMINI_INPUT_USD_PER_M = 2.00  # @param {type:"number"}
GEMINI_OUTPUT_USD_PER_M = 10.00  # @param {type:"number"}

# Allow environment overrides (handy for automated runs)
if PROJECT_ID == "your-gcp-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
SVI_PROJECT_ID = SVI_PROJECT_ID or os.environ.get("SVI_PROJECT_ID", "") or PROJECT_ID
assert PROJECT_ID != "your-gcp-project-id", "Set PROJECT_ID to your Google Cloud project."

bq_client = bigquery.Client(project=PROJECT_ID)
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_http = AuthorizedSession(creds)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)

print(f"Billing project: {PROJECT_ID} | Street View Insights: {SVI_PROJECT_ID}.{SVI_DATASET}")
print(f"Street center: ({CORRIDOR_LAT:.5f}, {CORRIDOR_LNG:.5f}), radius {CORRIDOR_RADIUS_M:.0f} m")

## 3. Step 1: Select images along one car drive

Each panoramic capture (`pano_id`) has 7 cameras: `_0`–`_5` look outward around the car (60° apart) and `_6` looks up at the sky. We keep only `_0`–`_5`.

A street usually contains captures from **several drives on different dates**. Mixing them can confuse triangulation (parked cars, new paint, slightly different positions), so we:

1. Query `pano_observations_latest` near the street with `ST_DWITHIN` (always use the `*_latest` view so image links are active).
2. Split captures into drives wherever two consecutive captures are more than 60 s apart.
3. Keep the drive with the most captures, then the `MAX_PANOS` captures closest to the street center.

The query is **dry-run first** and stops if it would scan more than `MAX_QUERY_GB`.

In [ ]:
# @title Query frames and select a single drive
corridor_sql = rf"""
SELECT
  observation_id,
  pano_id,
  capture_time,
  CAST(REGEXP_EXTRACT(observation_id, r'_(\d+):') AS INT64) AS cam_k,
  capture_location.latitude AS capture_lat,
  capture_location.longitude AS capture_lng,
  camera_pose.latitude AS cam_lat,
  camera_pose.longitude AS cam_lng,
  camera_pose.altitude AS cam_alt,
  camera_pose.heading AS heading,
  camera_pose.pitch AS pitch,
  camera_pose.roll AS roll,
  gcs_uri
FROM `{SVI_PROJECT_ID}.{SVI_DATASET}.pano_observations_latest`
WHERE pano_id IS NOT NULL
  AND ST_DWITHIN(
    ST_GEOGPOINT(capture_location.longitude, capture_location.latitude),
    ST_GEOGPOINT(@lng, @lat),
    @radius_m)
  AND CAST(REGEXP_EXTRACT(observation_id, r'_(\d+):') AS INT64) BETWEEN 0 AND 5
"""
query_params = [
    bigquery.ScalarQueryParameter("lat", "FLOAT64", CORRIDOR_LAT),
    bigquery.ScalarQueryParameter("lng", "FLOAT64", CORRIDOR_LNG),
    bigquery.ScalarQueryParameter("radius_m", "FLOAT64", CORRIDOR_RADIUS_M),
]

# 1. Dry run: how much will this scan?
dry_job = bq_client.query(
    corridor_sql,
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False, query_parameters=query_params),
)
gb = dry_job.total_bytes_processed / 1e9
print(f"Dry run: {gb:.2f} GB to scan (~${dry_job.total_bytes_processed / 1024**4 * 6.25:.3f} on-demand)")
if gb > MAX_QUERY_GB:
    raise RuntimeError(f"Query would scan {gb:.1f} GB > MAX_QUERY_GB={MAX_QUERY_GB}. Raise the limit if intended.")

# 2. Run the query with a hard billing cap
frames_all = bq_client.query(
    corridor_sql,
    job_config=bigquery.QueryJobConfig(
        query_parameters=query_params, maximum_bytes_billed=int(MAX_QUERY_GB * 1e9)
    ),
).to_dataframe()
if frames_all.empty:
    raise RuntimeError("No SV_PANO frames found here. Try another location or a larger radius.")


def select_single_drive(df: pd.DataFrame, max_panos: int, max_gap_s: float = 60.0) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split captures into drives by time gaps and keep the best drive's captures nearest the center."""
    panos = (
        df.groupby("pano_id")
        .agg(capture_time=("capture_time", "min"), lat=("capture_lat", "first"),
             lng=("capture_lng", "first"), n_cams=("cam_k", "nunique"))
        .reset_index()
    )
    panos = panos[panos["n_cams"] >= 6].sort_values("capture_time").reset_index(drop=True)
    gaps = panos["capture_time"].diff().dt.total_seconds().fillna(np.inf)
    panos["drive_id"] = (gaps > max_gap_s).cumsum()
    drives = panos.groupby("drive_id").agg(
        n_panos=("pano_id", "size"), start=("capture_time", "min"), end=("capture_time", "max")
    )
    best_drive = drives.sort_values(["n_panos", "end"], ascending=False).index[0]
    chosen = panos[panos["drive_id"] == best_drive].copy()
    # Equirectangular distance to the street center (accurate to cm at this scale)
    dy = (chosen["lat"] - CORRIDOR_LAT) * 111_320.0
    dx = (chosen["lng"] - CORRIDOR_LNG) * 111_320.0 * math.cos(math.radians(CORRIDOR_LAT))
    chosen["dist_m"] = np.hypot(dx, dy)
    keep_ids = chosen.nsmallest(max_panos, "dist_m")["pano_id"]
    out = df[df["pano_id"].isin(keep_ids)].sort_values(["capture_time", "cam_k"]).reset_index(drop=True)
    return out, drives.reset_index()


frames_df, drives_df = select_single_drive(frames_all, MAX_PANOS)
print(f"Found {frames_all['pano_id'].nunique()} captures from {len(drives_df)} drives; "
      f"using {frames_df['pano_id'].nunique()} captures ({len(frames_df)} frames) from one drive on "
      f"{frames_df['capture_time'].min():%Y-%m-%d}.")
drives_df.sort_values("n_panos", ascending=False).head(5)

## 4. Step 2: Find house features with Gemini

For every frame, Gemini returns structured JSON (enforced by a Pydantic schema) with one entry per visible street-facing building:

| Field | Meaning |
| :--- | :--- |
| `facade_box_2d` | Box around the front wall (`[ymin, xmin, ymax, xmax]`, 0–1000) |
| `front_door_box_2d` | Box around the front door, or `null` if hidden |
| `left/right_facade_corner_u_1000` | Horizontal position of the wall's two corners |
| `left/right_corner_visible` | `false` if a corner is cut off by the frame edge or hidden |
| `visible_house_number` | House number if readable on that building |
| `wall_material`, `roof_type`, `has_garage`, `has_porch`, `condition`, `stories`, `building_category` | **Asset features** (Objective 3). Edit the schema to detect your own features |

**Why the front door?** A door is a small, fixed point on the wall, so rays to it from different car positions meet tightly. A whole-house box changes shape as the car moves past.

Images are streamed into memory and sent as bytes. Each call retries with backoff on transient errors.

In [ ]:
# @title Gemini schema, prompt, and detection helper
class DetectedBuilding(BaseModel):
    facade_box_2d: list[int] = Field(description="[ymin, xmin, ymax, xmax] 0..1000 of the street-facing front wall.")
    front_door_box_2d: list[int] | None = Field(default=None, description="[ymin, xmin, ymax, xmax] 0..1000 of the main front door, or null if not visible.")
    left_facade_corner_u_1000: int = Field(description="Horizontal coordinate (0..1000) of the left vertical corner of the front wall.")
    right_facade_corner_u_1000: int = Field(description="Horizontal coordinate (0..1000) of the right vertical corner of the front wall.")
    left_corner_visible: bool = Field(description="True if the real left corner is visible (not cut by the image edge or hidden).")
    right_corner_visible: bool = Field(description="True if the real right corner is visible (not cut by the image edge or hidden).")
    building_category: str = Field(description="One of: DETACHED_SINGLE_FAMILY, SEMI_DETACHED_DUPLEX, TERRACED_ROWHOUSE, SMALL_MULTIFAMILY, COMMERCIAL_MIXED")
    stories: float = Field(description="Number of above-ground storeys, e.g. 1, 1.5, 2.")
    visible_house_number: str | None = Field(default=None, description="House number digits visibly mounted on this building or its porch, or null.")
    # ---- Asset features (edit / extend this block for your own use case) ----
    wall_material: str = Field(description="Main front-wall material: BRICK, WOOD_SIDING, VINYL_SIDING, STUCCO, STONE, CONCRETE, OTHER, or UNKNOWN.")
    roof_type: str = Field(description="Roof shape: GABLE, HIP, FLAT, OTHER, or NOT_VISIBLE.")
    has_garage: bool = Field(description="True if a garage door is visible on this building.")
    has_porch: bool = Field(description="True if a front porch or covered entry is visible.")
    condition: str = Field(description="Visible exterior condition: GOOD, FAIR, POOR, or UNKNOWN.")
    confidence: float = Field(description="Confidence 0.0-1.0.")


# Features aggregated per house across all its views (Objective 3). Add new schema fields here too.
FEATURE_FIELDS = ["building_category", "stories", "wall_material", "roof_type", "has_garage", "has_porch", "condition"]


class FrameDetections(BaseModel):
    buildings: list[DetectedBuilding] = Field(default_factory=list)


HOUSE_DETECTION_PROMPT = """
Detect every distinct street-facing building (houses, apartment buildings, shops) clearly visible in this street-level photo.
Rules:
1. Ignore buildings more than ~45 m away, sheds, fences, gas station canopies, and vehicles.
2. front_door_box_2d must be a real pedestrian door in the building wall. Never box a window, a sign, a gate, or a garage door.
   If the door is hidden by trees, bushes, or a car, return null.
3. Set left_corner_visible / right_corner_visible to false when that corner is cut off by the image edge or hidden.
4. Only report visible_house_number if the digits are physically on that building or its porch.
""".strip()

THINKING_BUDGET = None  # Leave None for the model's default; set an int only for models that accept a thinking budget.


def load_image_bytes(gcs_uri: str) -> bytes:
    """Stream a Cloud Storage object into memory (never written to disk)."""
    bucket, obj = gcs_uri.removeprefix("gs://").split("/", 1)
    url = f"https://storage.googleapis.com/storage/v1/b/{bucket}/o/{urllib.parse.quote(obj, safe='')}?alt=media"
    resp = authed_http.get(url, timeout=60)
    resp.raise_for_status()
    return resp.content


def _gemini_detect(image_bytes: bytes, max_attempts: int = 5):
    cfg = dict(temperature=0.0, response_mime_type="application/json", response_schema=FrameDetections)
    if THINKING_BUDGET is not None:
        cfg["thinking_config"] = types.ThinkingConfig(thinking_budget=THINKING_BUDGET)
    for attempt in range(max_attempts):
        try:
            return genai_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"), HOUSE_DETECTION_PROMPT],
                config=types.GenerateContentConfig(**cfg),
            )
        except Exception as e:  # retry only rate limits / transient server errors
            code_ = getattr(e, "code", None)
            retryable = code_ in (408, 429, 500, 502, 503, 504) or code_ is None
            if attempt == max_attempts - 1 or not retryable:
                raise
            wait = min(60.0, 2.0 ** attempt + random.random())
            print(f"  retry {attempt + 1} in {wait:.1f}s ({type(e).__name__}: {str(e)[:80]})")
            time.sleep(wait)


def _detect_one(row: Mapping[str, Any]) -> tuple[list[dict[str, Any]], dict[str, int]]:
    resp = _gemini_detect(load_image_bytes(row["gcs_uri"]))
    parsed = resp.parsed if isinstance(resp.parsed, FrameDetections) else FrameDetections.model_validate_json(resp.text or "{}")
    um = resp.usage_metadata
    usage = {
        "in": int(getattr(um, "prompt_token_count", 0) or 0),
        "out": int(getattr(um, "candidates_token_count", 0) or 0),
        "thinking": int(getattr(um, "thoughts_token_count", 0) or 0),
    }
    rows = [
        {"observation_id": row["observation_id"], "pano_id": row["pano_id"], "cam_k": int(row["cam_k"]),
         "gcs_uri": row["gcs_uri"], **b.model_dump()}
        for b in parsed.buildings
    ]
    return rows, usage


def detect_buildings_in_frames(df: pd.DataFrame, max_workers: int = 8) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """Run Gemini on every frame in parallel; returns raw detections and a usage/cost summary."""
    detections: list[dict[str, Any]] = []
    tot = {"in": 0, "out": 0, "thinking": 0}
    failed = 0
    with cf.ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(_detect_one, r) for r in df.to_dict("records")]
        for fut in cf.as_completed(futures):
            try:
                rows, usage = fut.result()
            except Exception as e:
                failed += 1
                print(f"  frame failed: {type(e).__name__}: {str(e)[:120]}")
                continue
            detections.extend(rows)
            for k in tot:
                tot[k] += usage[k]
    cost = (tot["in"] * GEMINI_INPUT_USD_PER_M + (tot["out"] + tot["thinking"]) * GEMINI_OUTPUT_USD_PER_M) / 1e6
    summary = {"frames": len(df), "failed_frames": failed, "building_boxes": len(detections),
               "input_tokens": tot["in"], "output_tokens": tot["out"], "thinking_tokens": tot["thinking"],
               "est_gemini_cost_usd": round(cost, 4)}
    return detections, summary


print("Detection helpers ready.")

In [ ]:
# @title Run Gemini on all selected frames
t0 = time.time()
raw_detections, gemini_usage = detect_buildings_in_frames(frames_df)
print(json.dumps(gemini_usage, indent=2))
print(f"Took {time.time() - t0:.0f}s. Frames with at least one building: "
      f"{len({d['observation_id'] for d in raw_detections})} / {len(frames_df)}")

## 5. Step 3a: From a pixel to a compass direction

To triangulate, every pixel must become a **direction in the world** (compass azimuth + elevation).

1. **Lens model.** `SV_PANO` frames are wide-angle, fisheye-style images, so straight-line (pinhole) math bends directions near the image edges. We use a simple **equidistant fisheye** model with one distortion term: the angle from the image center, $\theta$, maps to pixel radius $r = f\,\theta\,(1 + k_1\theta^2)$.
2. **Camera orientation.** `camera_pose.heading`, `pitch`, and `roll` rotate that direction from the camera into East-North-Up coordinates. Roll spins the image around the viewing direction.
3. **Camera position.** `camera_pose.latitude/longitude` already gives each camera's own position, so no extra offset is needed.

> The lens values below (`f_px`, `cx`, `cy`, `k1`) are **approximate values estimated from the imagery**, not official calibration data. The self-check below confirms basic behavior. You can refine them for your area, for example by checking that a distant landmark seen by two neighboring cameras gives the same bearing in both.

In [ ]:
# @title Lens model, camera rotation, and pixel -> bearing
CAM_HEIGHT_ABOVE_GROUND_M = 2.5  # approximate camera height above the road


@dataclass(frozen=True)
class FisheyeLens:
    width: int = 3648
    height: int = 5472
    f_px: float = 2847.13  # approximate equidistant focal length (pixels per radian)
    cx: float = 1835.6     # approximate optical center (pixels)
    cy: float = 2736.0
    k1: float = -0.028     # one radial distortion term


LENS = FisheyeLens()


def pixel_to_camera_ray(lens: FisheyeLens, u: float, v: float) -> np.ndarray:
    """Pixel -> unit direction in camera axes [right, down, forward] (equidistant fisheye, 1 term)."""
    mx, my = (u - lens.cx) / lens.f_px, (v - lens.cy) / lens.f_px
    theta_d = math.hypot(mx, my)
    theta = theta_d
    for _ in range(20):  # Newton solve of theta * (1 + k1 * theta^2) = theta_d
        f = theta * (1.0 + lens.k1 * theta * theta) - theta_d
        theta -= f / (1.0 + 3.0 * lens.k1 * theta * theta)
    s = math.sin(theta) / theta_d if theta_d > 1e-12 else 1.0
    return np.array([mx * s, my * s, math.cos(theta)])


def camera_to_enu_rotation(heading_deg: float, pitch_deg: float, roll_deg: float) -> np.ndarray:
    """3x3 matrix whose columns are the camera right, down, forward axes in East-North-Up."""
    h, p, r = map(math.radians, (heading_deg, pitch_deg, roll_deg))
    fwd = np.array([math.sin(h) * math.cos(p), math.cos(h) * math.cos(p), math.sin(p)])
    right0 = np.array([math.cos(h), -math.sin(h), 0.0])
    down0 = np.cross(fwd, right0)
    right = math.cos(r) * right0 + math.sin(r) * down0  # roll spins around the forward axis
    down = -math.sin(r) * right0 + math.cos(r) * down0
    return np.stack([right, down, fwd], axis=1)


def pixel_to_bearing(lens: FisheyeLens, heading: float, pitch: float, roll: float, u: float, v: float) -> tuple[float, float]:
    """Pixel + camera pose -> (azimuth deg clockwise from North, elevation deg)."""
    d = camera_to_enu_rotation(heading, pitch, roll) @ pixel_to_camera_ray(lens, u, v)
    az = math.degrees(math.atan2(d[0], d[1])) % 360.0
    el = math.degrees(math.asin(max(-1.0, min(1.0, d[2]))))
    return az, el


# ---- Self-check ----
for hdg in (0.0, 90.0, 225.0):
    az, el = pixel_to_bearing(LENS, hdg, 0.0, 0.0, LENS.cx, LENS.cy)
    assert abs(((az - hdg + 180) % 360) - 180) < 1e-6 and abs(el) < 1e-6, "image center must point along heading"
az_l, _ = pixel_to_bearing(LENS, 0.0, 0.0, 0.0, 0.0, LENS.cy)
az_r, _ = pixel_to_bearing(LENS, 0.0, 0.0, 0.0, LENS.width - 1, LENS.cy)
hfov = ((az_r - az_l) % 360.0)
print(f"Self-check OK. Horizontal field of view ~{hfov:.1f} deg per camera; "
      f"cameras are 60 deg apart, so neighbors overlap by ~{hfov - 60:.0f} deg.")

## 6. Step 3b: Group views of the same house and triangulate

Gemini gives unlabeled boxes per frame. It doesn't know that a box in frame A and one in frame B are the same house. We work that out from geometry:

1. **One ray per detection** to the front door (or, if the door is hidden, to a point along the wall between the two visible corners).
2. **Cannot-link rules.** Two detections are never merged if they show **different house numbers**, or if they are **two separate boxes in the same frame**.
3. **Seed and grow.** Intersect pairs of rays from different captures (at least 6° apart and at a plausible house distance), then add every other ray passing within **2.6 m** of that point.
4. **Triangulate only groups from 3+ car positions.** Two rays always cross *somewhere*, so a 2-position group can't check itself. Houses seen from 1–2 positions go to **Tier B** (Section 7b) with their images and an approximate location instead.
5. **Solve** each kept group with robust least squares (Huber IRLS) to get the **front door location**.
6. **Roof center.** Google Maps and footprint datasets place a house at the middle of its roof, not at the door. We move the point **inward**, perpendicular to the wall, by roughly half a house depth (a rule-of-thumb per building type, below), and output both points.

In [ ]:
# @title Grouping and triangulation solver
# Rule-of-thumb distance (m) from the front wall to the roof center, by building type. Tune for your area.
HALF_DEPTH_M = {
    "DETACHED_SINGLE_FAMILY": 6.2, "SEMI_DETACHED_DUPLEX": 5.4, "TERRACED_ROWHOUSE": 4.8,
    "SMALL_MULTIFAMILY": 6.8, "COMMERCIAL_MIXED": 7.5,
}
R_EARTH_M = 6_371_008.8


def lla_to_en(lat: float, lng: float, ref_lat: float, ref_lng: float) -> tuple[float, float]:
    """Lat/lng -> local East/North meters (fine for areas of a few hundred meters)."""
    east = math.radians(lng - ref_lng) * R_EARTH_M * math.cos(math.radians(0.5 * (lat + ref_lat)))
    north = math.radians(lat - ref_lat) * R_EARTH_M
    return east, north


def en_to_lla(east: float, north: float, ref_lat: float, ref_lng: float) -> tuple[float, float]:
    lat = ref_lat + math.degrees(north / R_EARTH_M)
    lng = ref_lng + math.degrees(east / (R_EARTH_M * math.cos(math.radians(0.5 * (lat + ref_lat)))))
    return lat, lng


def distance_m(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    e, n = lla_to_en(lat2, lng2, lat1, lng1)
    return math.hypot(e, n)


@dataclass
class HouseView:
    observation_id: str
    pano_id: str
    cam_k: int
    gcs_uri: str
    cam_lat: float
    cam_lng: float
    facade_box_px: tuple[float, float, float, float]
    door_box_px: tuple[float, float, float, float] | None
    u_ref_px: float
    v_ref_px: float
    keypoint_source: str
    u_left: float
    u_right: float
    left_visible: bool
    right_visible: bool
    alpha_door: float | None      # door position along the wall (0 = left corner, 1 = right corner)
    az_mid: float
    az_door: float | None
    az_left: float
    az_right: float
    est_range_m: float
    weight: float
    quality: float
    category: str
    house_number: str | None
    features: dict = dataclasses.field(default_factory=dict)
    frames: list = dataclasses.field(default_factory=list)  # all frames (cameras) of this position showing the house


def _box_to_px(box: Sequence[float] | None) -> tuple[float, float, float, float] | None:
    if not box or len(box) != 4:
        return None
    y0, x0, y1, x1 = (float(b) for b in box)
    px0, px1 = sorted((x0 / 1000 * LENS.width, x1 / 1000 * LENS.width))
    py0, py1 = sorted((y0 / 1000 * LENS.height, y1 / 1000 * LENS.height))
    px0, px1 = max(0.0, px0), min(float(LENS.width), px1)
    py0, py1 = max(0.0, py0), min(float(LENS.height), py1)
    return (px0, py0, px1, py1) if (px1 - px0) >= 6 and (py1 - py0) >= 6 else None


def build_house_view(det: Mapping[str, Any], pose: Mapping[str, Any]) -> HouseView | None:
    """One Gemini detection + its camera pose -> compass rays to the door and wall corners."""
    facade = _box_to_px(det.get("facade_box_2d"))
    if facade is None:
        return None
    door = _box_to_px(det.get("front_door_box_2d"))
    fx0, fy0, fx1, fy1 = facade
    fh = max(fy1 - fy0, 8.0)

    def _u(key: str, fallback: float) -> float:
        val = det.get(key)
        return float(np.clip(float(val) / 1000 * LENS.width, 0, LENS.width)) if val is not None else fallback

    u_left, u_right = _u("left_facade_corner_u_1000", fx0), _u("right_facade_corner_u_1000", fx1)
    if u_right <= u_left + 4:
        u_left, u_right = fx0, fx1
    left_vis = bool(det.get("left_corner_visible", True)) and u_left > 35
    right_vis = bool(det.get("right_corner_visible", True)) and u_right < LENS.width - 35

    hdg, pitch, roll = float(pose["heading"]), float(pose["pitch"]), float(pose["roll"])
    v_wall = fy0 + 0.78 * fh

    u_door = v_door = alpha = None
    source = "corners_midpoint" if (left_vis and right_vis) else "clipped_wall_fallback"
    if door is not None:
        u_door, v_door = 0.5 * (door[0] + door[2]), door[1] + 0.8 * (door[3] - door[1])
        source = "front_door_box"
        if left_vis and right_vis and (u_right - u_left) >= 20:
            alpha = float(np.clip((u_door - u_left) / (u_right - u_left), 0.05, 0.95))

    u_mid = 0.5 * (u_left + u_right)
    az_mid, el_base = pixel_to_bearing(LENS, hdg, pitch, roll, u_mid, fy1)
    az_l, _ = pixel_to_bearing(LENS, hdg, pitch, roll, u_left, v_wall)
    az_r, _ = pixel_to_bearing(LENS, hdg, pitch, roll, u_right, v_wall)
    az_d = pixel_to_bearing(LENS, hdg, pitch, roll, u_door, v_door)[0] if u_door is not None else None

    # Rough single-view distance: from apparent wall height, and from where the wall meets the ground
    stories = float(det.get("stories") or 2.0)
    d_size = max(3.6, 3.1 * stories + 1.5) / max(fh / LENS.f_px, 0.03)
    el = el_base - 0.75
    d_ground = CAM_HEIGHT_ABOVE_GROUND_M / math.tan(math.radians(-el)) if (-40 < el < -1.5 and fy1 < LENS.height - 15) else None
    rng = float(np.clip(0.5 * d_ground + 0.5 * d_size, 4.5, 45)) if (d_ground and 4 <= d_ground <= 38) else float(np.clip(d_size, 5, 45))

    conf = float(det.get("confidence") or 0.85)
    quality = (0.75 + 0.25 * conf) * (1.0 if (left_vis and right_vis) else 0.65)
    hnum = det.get("visible_house_number")
    hnum = str(hnum).strip() if hnum not in (None, "", "null", "None") else None
    return HouseView(
        observation_id=str(det["observation_id"]), pano_id=str(det["pano_id"]), cam_k=int(det["cam_k"]),
        gcs_uri=str(det.get("gcs_uri") or pose.get("gcs_uri") or ""),
        cam_lat=float(pose["cam_lat"]), cam_lng=float(pose["cam_lng"]),
        facade_box_px=facade, door_box_px=door,
        u_ref_px=float(u_door if u_door is not None else u_mid),
        v_ref_px=float(v_door if v_door is not None else v_wall),
        keypoint_source=source, u_left=u_left, u_right=u_right, left_visible=left_vis, right_visible=right_vis,
        alpha_door=alpha, az_mid=az_mid, az_door=az_d, az_left=az_l, az_right=az_r,
        est_range_m=rng, weight=quality / max(rng, 5.0), quality=quality,
        category=str(det.get("building_category") or "DETACHED_SINGLE_FAMILY").upper(), house_number=hnum,
        features={k: det.get(k) for k in FEATURE_FIELDS},
        frames=[str(det["observation_id"])],
    )


def _angle_between_lines(a: float, b: float) -> float:
    d = abs((a - b + 180) % 360 - 180)
    return min(d, 180 - d)


def _cannot_link(a: HouseView, b: HouseView) -> bool:
    if a.house_number and b.house_number and a.house_number != b.house_number:
        return True
    if a.pano_id == b.pano_id and a.cam_k == b.cam_k:
        return min(a.u_right, b.u_right) - max(a.u_left, b.u_left) <= 0  # separate boxes, same frame
    return False


def _solve_rays(pairs: list[tuple[HouseView, float]], ref_lat: float, ref_lng: float,
                min_parallax: float = 5.0, huber: float = 0.55):
    """Robust (Huber IRLS) intersection of 2D rays. Returns (xy, max_parallax, rms) or (None, ...)."""
    if len(pairs) < 2:
        return None, 0.0, float("inf")
    par = max(_angle_between_lines(pairs[i][1], pairs[j][1]) for i in range(len(pairs)) for j in range(i + 1, len(pairs)))
    if par < min_parallax:
        return None, par, float("inf")
    O = np.array([lla_to_en(v.cam_lat, v.cam_lng, ref_lat, ref_lng) for v, _ in pairs])
    D = np.array([(math.sin(math.radians(a)), math.cos(math.radians(a))) for _, a in pairs])
    N = np.array([(math.cos(math.radians(a)), -math.sin(math.radians(a))) for _, a in pairs])
    Q = np.array([v.quality for v, _ in pairs])
    b = np.sum(N * O, axis=1)
    w = np.array([v.weight for v, _ in pairs])
    xy = np.linalg.lstsq(N * w[:, None], b * w, rcond=None)[0]
    for _ in range(8):
        rel = xy - O
        perp = np.abs(np.sum(rel * N, axis=1))
        along = np.maximum(np.sum(rel * D, axis=1), 4.5)
        eff = np.sqrt(Q) / along ** 0.7 * np.sqrt(np.where(perp <= huber, 1.0, huber / np.maximum(perp, 1e-6)))
        nxt = np.linalg.lstsq(N * eff[:, None], b * eff, rcond=None)[0]
        done = np.linalg.norm(nxt - xy) < 1e-4
        xy = nxt
        if done:
            break
    along = np.sum((xy - O) * D, axis=1)
    if int(np.sum((along >= 2.5) & (along <= 65))) < 2:
        return None, par, float("inf")
    rms = float(np.sqrt(np.mean(np.sum((xy - O) * N, axis=1) ** 2)))
    return xy, par, rms


def triangulate_house(views: Sequence[HouseView], ref_lat: float, ref_lng: float, min_parallax: float = 5.0):
    """Front door + roof-center estimate for one group of views."""
    best_per_pano: dict[str, HouseView] = {}
    for v in views:
        if v.pano_id not in best_per_pano or v.weight > best_per_pano[v.pano_id].weight:
            best_per_pano[v.pano_id] = v
    work = list(best_per_pano.values())
    if len(work) < 2:
        return None

    xy_mid, par_mid, rms_mid = _solve_rays([(v, v.az_mid) for v in work], ref_lat, ref_lng, min_parallax, 1.1)
    if xy_mid is None:
        return None
    xy_l = _solve_rays([(v, v.az_left) for v in work if v.left_visible], ref_lat, ref_lng, min_parallax, 0.75)[0]
    xy_r = _solve_rays([(v, v.az_right) for v in work if v.right_visible], ref_lat, ref_lng, min_parallax, 0.75)[0]

    alphas = [v.alpha_door for v in work if v.alpha_door is not None]
    alpha_bar = float(np.median(alphas)) if alphas else 0.5
    door_pairs = []
    for v in work:
        if v.az_door is not None:
            door_pairs.append((v, v.az_door))
        elif v.left_visible and v.right_visible:  # door hidden: interpolate along the wall
            d_lr = ((v.az_right - v.az_left + 180) % 360) - 180
            door_pairs.append((dataclasses.replace(v, weight=0.6 * v.weight, quality=0.6 * v.quality),
                               (v.az_left + alpha_bar * d_lr) % 360))
    xy_door, par, rms = _solve_rays(door_pairs, ref_lat, ref_lng, min_parallax, 0.55)
    if xy_door is None:
        xy_door, par, rms = xy_mid.copy(), par_mid, rms_mid

    # Inward direction: perpendicular to the road (from the camera track), refined by the wall if both corners solved
    cams = np.array([lla_to_en(v.cam_lat, v.cam_lng, ref_lat, ref_lng) for v in work])
    view_dir = xy_door - cams.mean(axis=0)
    view_dir /= max(np.linalg.norm(view_dir), 1e-6)
    n_in = view_dir.copy()
    if len(cams) >= 2:
        t_road = np.linalg.svd(cams - cams.mean(axis=0), full_matrices=False)[2][0]
        n_in = np.array([-t_road[1], t_road[0]])
        n_in = n_in if n_in @ view_dir >= 0 else -n_in
    width = None
    if xy_l is not None and xy_r is not None and 3.2 <= np.linalg.norm(xy_r - xy_l) <= 28:
        seg = xy_r - xy_l
        width = float(np.linalg.norm(seg))
        n_wall = np.array([-seg[1], seg[0]]) / width
        n_wall = n_wall if n_wall @ view_dir >= 0 else -n_wall
        if n_wall @ n_in >= math.cos(math.radians(32)):
            n_in = 0.55 * n_wall + 0.45 * n_in
            n_in /= np.linalg.norm(n_in)
    if width is None:
        est = [2 * np.linalg.norm(xy_door - c) * math.tan(0.5 * math.radians(_angle_between_lines(v.az_right, v.az_left)))
               for v, c in zip(work, cams)]
        width = float(np.clip(np.median(est), 4.0, 22.0))

    t_wall = np.array([n_in[1], -n_in[0]])
    if view_dir[0] * t_wall[1] - view_dir[1] * t_wall[0] > 0:
        t_wall = -t_wall
    door_to_mid = float(np.clip((0.5 - alpha_bar) * width, -4.5, 4.5))
    wall_center = 0.65 * (xy_door + door_to_mid * t_wall) + 0.35 * xy_mid
    cats = [v.category for v in work]
    category = max(set(cats), key=cats.count)
    depth = float(np.clip(HALF_DEPTH_M.get(category, 5.8) + np.clip(0.15 * (width - 9.0), -1.0, 1.5), 2.5, 11.0))
    xy_roof = wall_center + depth * n_in

    ent_lat, ent_lng = en_to_lla(*xy_door, ref_lat, ref_lng)
    roof_lat, roof_lng = en_to_lla(*xy_roof, ref_lat, ref_lng)
    nums = [v.house_number for v in work if v.house_number]
    best_num = max(set(nums), key=nums.count) if nums else None
    # Objective 3: combine each feature across views (majority vote + how many views agree)
    features, agreement = {}, {}
    for key in FEATURE_FIELDS:
        vals = [str(v.features.get(key)) for v in work if v.features.get(key) not in (None, "", "UNKNOWN", "NOT_VISIBLE")]
        if vals:
            top = max(set(vals), key=vals.count)
            features[key], agreement[key] = top, f"{vals.count(top)}/{len(work)}"
        else:
            features[key], agreement[key] = None, f"0/{len(work)}"
    return {
        "entrance_lat": ent_lat, "entrance_lng": ent_lng, "centroid_lat": roof_lat, "centroid_lng": roof_lng,
        "n_views": len(work), "n_door_views": sum(v.door_box_px is not None for v in work),
        "n_number_views": nums.count(best_num) if best_num else 0, "house_number": best_num,
        "max_parallax_deg": round(par, 1), "ray_rms_m": round(rms, 2), "facade_width_m": round(width, 1),
        "depth_shift_m": round(depth, 1), "building_category": category, "views": work,
        "features": features, "feature_agreement": agreement,
        "n_images": sum(len(v.frames) for v in work),
    }


DIST_BUCKETS_M = (5, 10, 20, 30, 45)


def approximate_location(views: Sequence[HouseView], ref_lat: float, ref_lng: float) -> dict[str, Any]:
    """Rough house location when triangulation isn't possible: car position + bearing + single-view distance."""
    pts, cams = [], []
    for v in views:
        e, n = lla_to_en(v.cam_lat, v.cam_lng, ref_lat, ref_lng)
        az = math.radians(v.az_door if v.az_door is not None else v.az_mid)
        pts.append((e + v.est_range_m * math.sin(az), n + v.est_range_m * math.cos(az)))
        cams.append((e, n))
    P = np.array(pts)
    c = P.mean(axis=0)
    spread = float(np.max(np.linalg.norm(P - c, axis=1))) if len(P) > 1 else 0.0
    d_car = float(np.min(np.linalg.norm(np.array(cams) - c, axis=1)))
    lat, lng = en_to_lla(float(c[0]), float(c[1]), ref_lat, ref_lng)
    return {"approx_lat": lat, "approx_lng": lng, "nearest_car_m": round(d_car, 1),
            "within_m_of_car": next((b for b in DIST_BUCKETS_M if d_car <= b), 60),
            "uncertainty_m": round(spread + 0.25 * d_car, 1)}


def _candidate(group: Sequence[HouseView], reason: str, ref_lat: float, ref_lng: float) -> dict[str, Any]:
    nums = [v.house_number for v in group if v.house_number]
    return {"n_positions": len({v.pano_id for v in group}), "n_images": sum(len(v.frames) for v in group),
            "house_number": nums[0] if nums else None, "reason": reason,
            "observation_ids": [o for v in group for o in v.frames], **approximate_location(group, ref_lat, ref_lng)}


def solve_houses_corridor(frames: pd.DataFrame, detections: Sequence[Mapping[str, Any]], min_views: int = 3,
                          min_parallax: float = 6.0, gate_m: float = 2.6, dedup_m: float = 4.2):
    """Group unlabeled detections into houses.

    Returns (houses, approx_candidates, view_stats):
      houses             - groups seen from >= min_views car positions, triangulated to a lat/lng
      approx_candidates  - houses seen from fewer positions (or whose solve failed): not triangulated, but
                           still with all their images and an approximate location from the car positions
      view_stats         - counts of how detections were used
    """
    pose_by_obs = {r["observation_id"]: r for r in frames.to_dict("records")}
    views = [hv for d in detections if (p := pose_by_obs.get(d["observation_id"])) and (hv := build_house_view(d, p))]
    stats = {"detections": len(detections), "usable_views": len(views), "views_in_houses": 0,
             "views_in_set_aside": 0, "views_ungrouped": 0}
    if not views:
        return [], [], stats
    ref_lat, ref_lng = float(frames["cam_lat"].mean()), float(frames["cam_lng"].mean())

    # Same capture seen by two neighboring cameras: keep the stronger view as the ray, remember the extra image
    dedup: list[HouseView] = []
    for pid in {v.pano_id for v in views}:
        kept: list[HouseView] = []
        for v in sorted((x for x in views if x.pano_id == pid), key=lambda x: -x.weight):
            az = v.az_door if v.az_door is not None else v.az_mid
            twin = next((u for u in kept if u.cam_k != v.cam_k and not _cannot_link(u, v)
                         and _angle_between_lines(az, u.az_door if u.az_door is not None else u.az_mid) < 6), None)
            if twin is None:
                kept.append(v)
            else:
                twin.frames.extend(v.frames)
        dedup.extend(kept)
    n = len(dedup)
    AZ = [v.az_door if v.az_door is not None else v.az_mid for v in dedup]
    O = np.array([lla_to_en(v.cam_lat, v.cam_lng, ref_lat, ref_lng) for v in dedup])
    D = np.array([(math.sin(math.radians(a)), math.cos(math.radians(a))) for a in AZ])
    N = np.array([(math.cos(math.radians(a)), -math.sin(math.radians(a))) for a in AZ])
    W = np.array([v.weight for v in dedup])
    M = np.array([v.est_range_m for v in dedup])
    P = [v.pano_id for v in dedup]

    seeds = []
    for i in range(n):
        for j in range(i + 1, n):
            if P[i] == P[j] or _cannot_link(dedup[i], dedup[j]):
                continue
            if not 2.5 <= np.linalg.norm(O[i] - O[j]) <= 65 or _angle_between_lines(AZ[i], AZ[j]) < min_parallax:
                continue
            try:
                pt = np.linalg.solve(np.vstack([N[i], N[j]]), [N[i] @ O[i], N[j] @ O[j]])
            except np.linalg.LinAlgError:
                continue
            ti, tj = (pt - O[i]) @ D[i], (pt - O[j]) @ D[j]
            if 3.5 <= ti <= 48 and 3.5 <= tj <= 48 and 0.35 <= ti / M[i] <= 2.6 and 0.35 <= tj / M[j] <= 2.6:
                seeds.append((pt, i, j))

    used = np.zeros(n, dtype=bool)
    houses, set_aside = [], []
    while True:
        best = None
        for pt, i, j in seeds:
            if used[i] or used[j]:
                continue
            rel = pt - O
            along = np.sum(rel * D, axis=1)
            perp = np.abs(np.sum(rel * N, axis=1))
            ok = (~used) & (along >= 3.5) & (along <= 48) & (perp <= gate_m) & (along / M >= 0.35) & (along / M <= 2.6)
            per_pano: dict[str, int] = {}
            for k in map(int, np.nonzero(ok)[0]):
                if _cannot_link(dedup[i], dedup[k]) or _cannot_link(dedup[j], dedup[k]):
                    continue
                if P[k] not in per_pano or perp[k] < perp[per_pano[P[k]]]:
                    per_pano[P[k]] = k
            inl = list(per_pano.values())
            if len(inl) < 2:
                continue
            key = (len(inl) >= 3, float(np.sum(W[inl] * (1 - 0.35 * perp[inl] / gate_m))))
            if best is None or key > best[0]:
                best = (key, inl)
        if best is None:
            break
        inl = best[1]
        used[inl] = True
        group = [dedup[k] for k in inl]
        if len(inl) < min_views:
            # Too few car positions to triangulate reliably: keep images + approximate location
            set_aside.append(_candidate(group, f"only {len(inl)} car positions", ref_lat, ref_lng))
            stats["views_in_set_aside"] += len(inl)
            continue
        h = triangulate_house(group, ref_lat, ref_lng, min_parallax)
        if h is not None and h["n_views"] >= min_views and h["ray_rms_m"] <= 1.85:
            houses.append(h)
            stats["views_in_houses"] += len(inl)
        else:
            set_aside.append(_candidate(group, "rays did not converge", ref_lat, ref_lng))
            stats["views_in_set_aside"] += len(inl)
    # Seen from a single car position (possibly by 2 cameras): still an image set + rough location
    for k in map(int, np.nonzero(~used)[0]):
        set_aside.append(_candidate([dedup[k]], "only 1 car position", ref_lat, ref_lng))
    stats["views_ungrouped"] = int((~used).sum())
    stats["same_capture_duplicates"] = len(views) - n

    kept_houses: list[dict[str, Any]] = []
    for h in sorted(houses, key=lambda x: (-x["n_views"], x["ray_rms_m"])):
        if any(not (h["house_number"] and k["house_number"] and h["house_number"] != k["house_number"])
               and distance_m(h["entrance_lat"], h["entrance_lng"], k["entrance_lat"], k["entrance_lng"]) < dedup_m
               for k in kept_houses):
            continue
        h["house_id"] = f"H{len(kept_houses) + 1:02d}"
        kept_houses.append(h)
    for i, s in enumerate(sorted(set_aside, key=lambda s: (-s["n_positions"], -s["n_images"]))):
        s["candidate_id"] = f"A{i + 1:02d}"
    set_aside.sort(key=lambda s: s["candidate_id"])
    # A Tier B candidate whose rough location overlaps a triangulated house is probably the same house
    for s in set_aside:
        near = [(distance_m(s["approx_lat"], s["approx_lng"], h["centroid_lat"], h["centroid_lng"]), h["house_id"])
                for h in kept_houses]
        d, hid = min(near) if near else (None, None)
        s["likely_same_as"] = hid if d is not None and d <= max(8.0, s["uncertainty_m"]) else None
    return kept_houses, set_aside, stats


print("Solver ready.")

## 7. Results in two tiers

Every house Gemini found ends up in one of two tiers. Both are useful.

| Tier | When | What you get |
| :--- | :--- | :--- |
| **A. Triangulated** | Seen from **3+ car positions** | A lat/lng from intersecting rays (roughly meter-level) + all its images + features |
| **B. Approximate** | Seen from 1–2 car positions, or rays didn't agree | All its images + features + an **approximate location**: "within 5 / 10 / 20 / 30 / 45 m of this car position" |

**Positions vs. images.** The cameras on one capture sit only centimeters apart, so they count as **one car position** for triangulation. Each camera frame that shows the house still counts as an **image**. A house seen by two cameras at one position has 2 images but 1 position.

Tier B is not discarded. It tells you a house exists, gives you good images of it, and gives a rough location from the car's position, bearing, and the house's apparent size. We track how many houses land in Tier B so we can work on moving them to Tier A (more captures, more drives, better door detection).

In [ ]:
# @title 7a. Tier A: houses triangulated from 3+ car positions
MIN_VIEWS = 3


def quality_checks(h: Mapping[str, Any]) -> dict[str, Any]:
    flags = []
    if h["ray_rms_m"] > 0.5:
        flags.append("rays disagree")
    if h["n_door_views"] < 1:
        flags.append("door never seen")
    return {"quality": "HIGH" if not flags else "REVIEW", "flags": ", ".join(flags) or "-"}


houses, approx_houses, view_stats = solve_houses_corridor(frames_df, raw_detections, min_views=MIN_VIEWS)
set_aside = approx_houses  # kept for backward compatibility with earlier versions
for h in houses:
    h.update(quality_checks(h))

HOUSE_COLUMNS = ["house_id", "n_views", "n_images", "n_door_views", "house_number", "centroid_lat", "centroid_lng",
                 "ray_rms_m", "max_parallax_deg", "quality", "flags"]
houses_df = pd.DataFrame(
    [{**{c: h.get(c) for c in HOUSE_COLUMNS},
      **{f: f"{h['features'][f]} ({h['feature_agreement'][f]})" for f in FEATURE_FIELDS}} for h in houses],
    columns=HOUSE_COLUMNS + FEATURE_FIELDS,
).rename(columns={"n_views": "n_positions"})
print(f"Tier A: {len(houses_df)} houses triangulated from {MIN_VIEWS}+ car positions "
      f"({(houses_df['quality'] == 'HIGH').sum()} pass both checks). "
      "Feature cells show the majority value and how many positions agree.")
houses_df

In [ ]:
# @title 7b. Tier B: houses with images + approximate location (not triangulated)
approx_df = pd.DataFrame(
    [{"candidate_id": s["candidate_id"], "n_positions": s["n_positions"], "n_images": s["n_images"],
      "reason": s["reason"], "within_m_of_car": s["within_m_of_car"], "uncertainty_m": s["uncertainty_m"],
      "approx_lat": round(s["approx_lat"], 6), "approx_lng": round(s["approx_lng"], 6),
      "house_number": s["house_number"], "likely_same_as": s["likely_same_as"],
      "images": ", ".join(o.split(":")[1] for o in s["observation_ids"])}
     for s in approx_houses],
    columns=["candidate_id", "n_positions", "n_images", "reason", "within_m_of_car", "uncertainty_m",
             "approx_lat", "approx_lng", "house_number", "likely_same_as", "images"],
)
n_a, n_b = len(houses), len(approx_houses)
n_b_multi = int((approx_df["n_images"] >= 2).sum())
n_b_dup = int(approx_df["likely_same_as"].notna().sum())
print(f"Tier B: {n_b} houses not triangulated ({n_b_dup} overlap a Tier A house and are probably extra images of it)")
for reason, cnt in approx_df["reason"].value_counts().items():
    print(f"  {reason:<28} {cnt}")
print(f"  ...of which have 2+ images: {n_b_multi} | a single image: {n_b - n_b_multi}")
if n_b:
    buckets = approx_df["within_m_of_car"].value_counts().sort_index()
    print("  Approximate distance from the nearest car position: "
          + ", ".join(f"<= {int(b)} m: {c}" for b, c in buckets.items()))
print(f"\nDetections used: {view_stats['detections']} total | {view_stats.get('same_capture_duplicates', 0)} extra camera "
      f"images of the same position | {view_stats['views_in_houses']} in Tier A | "
      f"{view_stats['views_in_set_aside'] + view_stats['views_ungrouped']} in Tier B")
approx_df

## 8. Step 4: Sanity check and summary of the three objectives

The model never saw any footprint or address. As a **light sanity check** (not an accuracy benchmark), we ask whether each located house sits on or near a real building and resolves to a real address. A few meters of error is expected and fine.

1. **Overture Maps building footprints.** Download only the buildings in a small box around the street directly from Overture's public release, using the open-source [`overturemaps`](https://github.com/OvertureMaps/overturemaps-py) package. Nothing is pre-packaged; you can repeat this for any area. (Querying the full Overture table in BigQuery for a small area scans hundreds of GB, so we avoid it.)
2. **Nearest building.** Distance from each located house to the nearest footprint edge (0 m means inside a building).
3. **Address (optional).** Reverse-geocode each located house with the [Geocoding API](https://developers.google.com/maps/documentation/geocoding) to get its `place_id` and address, and check whether a house number Gemini read matches it.

In [ ]:
# @title Download Overture building footprints for this street (live, public data)
from overturemaps import core as overture

pad_m = CORRIDOR_RADIUS_M + 40.0
dlat = pad_m / 111_320.0
dlng = pad_m / (111_320.0 * math.cos(math.radians(CORRIDOR_LAT)))
bbox = (CORRIDOR_LNG - dlng, CORRIDOR_LAT - dlat, CORRIDOR_LNG + dlng, CORRIDOR_LAT + dlat)

reader = overture.record_batch_reader("building", bbox=bbox)
table = reader.read_all() if reader is not None else None

footprints = []
if table is not None and table.num_rows:
    for gers_id, wkb in zip(table.column("id").to_pylist(), table.column("geometry").to_pylist()):
        geom = shapely.from_wkb(wkb)
        if isinstance(geom, MultiPolygon):
            geom = max(geom.geoms, key=lambda g: g.area)
        if isinstance(geom, Polygon):
            footprints.append({"overture_id": gers_id, "geometry": geom})
gt_df = pd.DataFrame(footprints, columns=["overture_id", "geometry"])
print(f"Downloaded {len(gt_df)} Overture building footprints around the street "
      f"(release {overture.get_latest_release()}). © Overture Maps Foundation, © OpenStreetMap contributors (ODbL).")

In [ ]:
# @title Sanity check each house and summarize the three objectives
NEAR_BUILDING_M = 5.0  # "on or near a building" tolerance


def reverse_geocode(lat: float, lng: float) -> dict[str, Any] | None:
    # Geocoding API reverse lookup -> best result (place_id, address, location, granularity).
    resp = authed_http.get(
        f"https://geocode.googleapis.com/v4beta/geocode/location/{lat},{lng}",
        headers={"X-Goog-User-Project": PROJECT_ID}, timeout=20,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Geocoding API error {resp.status_code}: {resp.text[:200]}")
    results = resp.json().get("results", [])
    if not results:
        return None
    r = results[0]
    return {"place_id": r.get("placeId"), "address": r.get("formattedAddress"), "granularity": r.get("granularity")}


def dist_to_nearest_building_m(lat: float, lng: float) -> tuple[float | None, int | None]:
    # Meters from a point to the nearest footprint (0 if inside). Uses a local meter grid.
    if gt_df.empty:
        return None, None
    p = Point(0.0, 0.0)
    best_d, best_j = None, None
    for j, g in enumerate(gt_df["geometry"]):
        ring = Polygon([lla_to_en(y, x, lat, lng) for x, y in g.exterior.coords])
        d = 0.0 if ring.contains(p) else ring.exterior.distance(p)
        if best_d is None or d < best_d:
            best_d, best_j = d, j
    return round(best_d, 1), best_j


geocode_ok = ENABLE_ADDRESS_CHECK
records, nearest_idx = [], {}
for i, h in enumerate(houses):
    d_bld, j = dist_to_nearest_building_m(h["centroid_lat"], h["centroid_lng"])
    if j is not None:
        nearest_idx[i] = j
    rec = {"house_id": h["house_id"], "n_views": h["n_views"], "lat": round(h["centroid_lat"], 6),
           "lng": round(h["centroid_lng"], 6), "ray_rms_m": h["ray_rms_m"], "dist_to_building_m": d_bld,
           "house_number_read": h["house_number"], "place_id": None, "address": None, "number_matches_address": None}
    if geocode_ok:
        try:
            addr = reverse_geocode(h["centroid_lat"], h["centroid_lng"])
        except RuntimeError as e:
            print(f"Address check disabled: {e}\n(Enable the Geocoding API in {PROJECT_ID} or set ENABLE_ADDRESS_CHECK=False.)")
            geocode_ok, addr = False, None
        if addr:
            rec.update(place_id=addr["place_id"], address=addr["address"])
            if h["house_number"] and addr["address"]:
                rec["number_matches_address"] = addr["address"].split(" ")[0] == h["house_number"]
    records.append(rec)
verif_df = pd.DataFrame(records, columns=["house_id", "n_views", "lat", "lng", "ray_rms_m", "dist_to_building_m",
                                          "house_number_read", "place_id", "address", "number_matches_address"])

n_located = len(houses)
set_aside_new = [s for s in set_aside if not s.get("likely_same_as")]
n_overlap = len(set_aside) - len(set_aside_new)
n_aside = len(set_aside_new)
n_cand = n_located + n_aside
n_multi_img = n_located + sum(s["n_images"] >= 2 for s in set_aside_new)
print("Objective 1 - Find each house in multiple images")
print(f"  Distinct houses found: {n_cand}  ({n_overlap} more Tier B candidates overlap a Tier A house and aren't counted again)")
if n_cand:
    print(f"  In 2+ images: {n_multi_img} ({n_multi_img / n_cand:.0%}) | "
          f"from {MIN_VIEWS}+ car positions (Tier A): {n_located} ({n_located / n_cand:.0%})")
    print(f"  Tier B (not triangulated): {n_aside} -> "
          f"{sum(s['n_images'] >= 2 for s in set_aside_new)} still have 2+ images, "
          f"{sum(s['n_positions'] == 2 for s in set_aside_new)} were seen from exactly 2 car positions")
if n_located:
    print(f"  Tier A per house: median {int(np.median([h['n_views'] for h in houses]))} car positions, "
          f"{int(np.median([h['n_images'] for h in houses]))} images")
print("Objective 2 - Locate each house")
print(f"  Tier A (triangulated lat/lng): {n_located}")
if n_aside:
    within = pd.Series([s["within_m_of_car"] for s in set_aside_new]).value_counts().sort_index()
    print(f"  Tier B (approximate, from car position): {n_aside} -> "
          + ", ".join(f"within {int(b)} m: {c}" for b, c in within.items()))
if n_located:
    print(f"  Median ray RMS: {np.median([h['ray_rms_m'] for h in houses]):.2f} m (how tightly the rays meet)")
    near = verif_df["dist_to_building_m"].dropna()
    if len(near):
        print(f"  On or within {NEAR_BUILDING_M:.0f} m of a building footprint: {(near <= NEAR_BUILDING_M).mean():.0%} "
              f"(median {near.median():.1f} m)")
    if verif_df["address"].notna().any():
        print(f"  Resolved to a street address: {verif_df['address'].notna().mean():.0%}")
    nm = verif_df["number_matches_address"].dropna()
    if len(nm):
        print(f"  House number read by Gemini matches the address: {int(nm.sum())}/{len(nm)}")
print("Objective 3 - Detect features")
if n_located:
    for f in FEATURE_FIELDS:
        agree = [int(h["feature_agreement"][f].split("/")[0]) / h["n_views"] for h in houses if h["features"][f] is not None]
        found = sum(h["features"][f] is not None for h in houses)
        print(f"  {f:<18} detected for {found}/{n_located} houses, average view agreement "
              + (f"{np.mean(agree):.0%}" if agree else "-"))
print("Addresses: Google Maps. Footprints: © Overture Maps Foundation, © OpenStreetMap contributors.")
verif_df

## 9. See one house end-to-end

Top: up to three frames that were grouped as the same house, with the wall box (dashed), door box (green), and the point used for the ray (red). Bottom: a top-down map with the camera positions, the rays, the solved door, the shift to the roof center, and the Overture footprint.

In [ ]:
# @title Plot the best-supported house
if not houses:
    print("No houses located on this street. Try another location or a larger MAX_PANOS.")
else:
    best = max(range(len(houses)), key=lambda i: (houses[i]["n_views"], -houses[i]["ray_rms_m"]))
    h = houses[best]
    v_row = verif_df.iloc[best]
    shown = h["views"][:3]

    fig = plt.figure(figsize=(15, 9), dpi=110)
    grid = fig.add_gridspec(2, 3, height_ratios=[1.0, 1.1], hspace=0.3, wspace=0.12)
    for col, v in enumerate(shown):
        ax = fig.add_subplot(grid[0, col])
        img = Image.open(io.BytesIO(load_image_bytes(v.gcs_uri)))
        fx0, fy0, fx1, fy1 = v.facade_box_px
        cx_, cy_ = 0.5 * (fx0 + fx1), 0.5 * (fy0 + fy1)
        bw = max(fx1 - fx0, 1100) * 1.6
        bh = bw / 1.45
        x1, x2 = max(0, int(cx_ - bw / 2)), min(img.width, int(cx_ + bw / 2))
        y1, y2 = max(0, int(cy_ - bh / 2)), min(img.height, int(cy_ + bh / 2))
        ax.imshow(img.crop((x1, y1, x2, y2)))
        ax.add_patch(mpatches.Rectangle((fx0 - x1, fy0 - y1), fx1 - fx0, fy1 - fy0, fill=False, ec="#38BDF8", lw=2, ls="--"))
        if v.door_box_px:
            dx0, dy0, dx1, dy1 = v.door_box_px
            ax.add_patch(mpatches.Rectangle((dx0 - x1, dy0 - y1), dx1 - dx0, dy1 - dy0, fill=False, ec="#22C55E", lw=2.4))
        ax.scatter([v.u_ref_px - x1], [v.v_ref_px - y1], s=60, c="#EF4444", ec="white", zorder=5)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"View {col + 1}: camera {v.cam_k} | {v.keypoint_source}", fontsize=9)
        ax.text(0.98, 0.03, "Imagery © Google", transform=ax.transAxes, ha="right", fontsize=7.5, color="white",
                bbox={"facecolor": "black", "alpha": 0.55, "pad": 2, "edgecolor": "none"})

    ax = fig.add_subplot(grid[1, :])
    ref_lat, ref_lng = h["entrance_lat"], h["entrance_lng"]
    if best in nearest_idx:
        poly = gt_df.iloc[nearest_idx[best]]["geometry"]
        xy = [lla_to_en(lat, lng, ref_lat, ref_lng) for lng, lat in poly.exterior.coords]
        ax.add_patch(mpatches.Polygon(xy, closed=True, fc="#DCFCE7", ec="#16A34A", lw=2, alpha=0.8, label="Nearest Overture footprint"))
    for v in h["views"]:
        ce, cn = lla_to_en(v.cam_lat, v.cam_lng, ref_lat, ref_lng)
        ax.plot([ce, 0], [cn, 0], c="#0284C7", lw=1.5, alpha=0.8)
        ax.scatter([ce], [cn], s=40, c="#1D4ED8", zorder=4)
    re_, rn_ = lla_to_en(h["centroid_lat"], h["centroid_lng"], ref_lat, ref_lng)
    ax.annotate("", xy=(re_, rn_), xytext=(0, 0), arrowprops={"arrowstyle": "-|>", "color": "#DC2626", "lw": 2})
    ax.scatter([0], [0], s=90, c="#F59E0B", ec="white", zorder=6, label="Solved front door")
    ax.scatter([re_], [rn_], s=170, marker="*", c="#EF4444", ec="white", zorder=7, label=f"Roof center (+{h['depth_shift_m']} m)")
    label = v_row["address"] or h["house_id"]
    feats = ", ".join(f"{k}={h['features'][k]}" for k in ("wall_material", "roof_type", "stories") if h["features"][k])
    ax.set_title(f"{label} | {h['n_views']} views | ray RMS {h['ray_rms_m']:.2f} m | {feats}", fontsize=10)
    ax.set_xlabel("East (m)"); ax.set_ylabel("North (m)")
    ax.axis("equal"); ax.grid(ls="--", alpha=0.35); ax.legend(loc="upper right", fontsize=8)
    ax.text(0.01, 0.02, "Footprint © Overture Maps Foundation, © OpenStreetMap contributors | Address: Google Maps",
            transform=ax.transAxes, fontsize=7.5, color="#475569")
    plt.show()

## 10. Adapting this to your data

* **Another street:** change `CORRIDOR_LAT/LNG`. Dense urban streets and rowhouses benefit from more captures (`MAX_PANOS`) and readable house numbers.
* **Other assets:** the same pixel → ray → group → solve pattern works for poles, signs, hydrants, or trees. Change the Gemini schema and the "point to aim at" (for example, the base of a pole).
* **Tuning:** `HALF_DEPTH_M` (wall-to-roof-center distance), the 2.6 m grouping gate, and the quality thresholds are starting points. Check them against footprints in your area.
* **Scale:** for many streets, use the [Vertex AI batch prediction API](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/batch-prediction-gemini) for Gemini and run each street independently.